
# FINAL-FREEZE-Erweiterung: FPR, AP-Korrektur, Systemvergleich, Kaskade und Fehleranalyse

Dieses Notebook **setzt den abgeschlossenen FINAL FREEZE fort**. Es ersetzt ihn nicht.

## Ziele

1. **False-Positive-Rate vollständig ausweisen**
   - Ziel-FPR: **0,5 %, 1,0 %, 2,0 %**
   - für jede schwellenwertabhängige Ergebniszeile:
     - `target_fpr`
     - `calibration_empirical_fpr`
     - `empirical_fpr`
     - FP/TN/TP/FN

2. **AP-Vergleich IID ↔ Distribution Shift korrigieren**
   - das bisherige `IID_ORIGINAL` bleibt für operative Kennzahlen erhalten;
   - für AP wird zusätzlich ein deterministisches **50/50-IID-Set** gebildet;
   - alle bestehenden Stresssets sind ebenfalls 50/50;
   - **AP-Degradation wird ausschließlich gegen `IID_BALANCED` berechnet**;
   - zusätzlich wird als Sensitivitätsanalyse für alle Szenarien ein **gleich großes AP-Set**
     mit derselben Klassenverteilung erzeugt (`AP_EQUAL_N`).

   Wichtig: Die zentrale Korrektur ist die **gleiche Klassenprävalenz**. Eine identische
   Gesamtgröße ist für AP nicht erforderlich; die Equal-N-Auswertung dient nur als zusätzliche
   Robustheitsprüfung.

3. **FINAL FREEZE mit den vorhandenen Embeddings neu auswerten**
   - BASE / DAPT / CONTRASTIVE
   - Logistic Regression / XGBoost / MLP
   - 10 % / 25 % / 100 % Labels
   - fünf Seeds
   - IID + Temporal + Domain-OOD + Template-OOD + Domain+Template-OOD

4. **Systemebene bei 25 % Labels ergänzen**
   - `B0_STRUCT_XGB`: XGBoost auf expliziten URL-/HTML-Strukturmerkmalen
   - `T0_E2E`: RoBERTa, direkt supervised fine-getuned
   - `DAPT_E2E`: gleicher Fine-Tuning-Ablauf, Initialisierung aus DAPT-40k
   - `BASE_EMB_MLP`: Repräsentationskontrolle
   - `DAPT_EMB_MLP`: stärkere DAPT-Embedding-Systemvariante
   - `CONTRASTIVE_EMB_MLP`: negative/komplementäre Referenz

5. **Kaskade**
   - Stage 1: `B0_STRUCT_XGB` blockiert am kalibrierten FPR-Arbeitspunkt;
   - Stage 2: Ranking der **nicht blockierten** Seiten;
   - Reviewbudgets 5 %, 10 %, 20 %;
   - Stage 2 ist Review/Triage und **keine automatische Blockentscheidung**.

6. **Fehleranalyse**
   - B0 vs. T0 / DAPT-E2E / DAPT-Embedding / Kontrollen
   - Rescue, Regression, FN-/FP-Überlappung
   - jeweils inklusive der tatsächlich beobachteten FPR.

## Leakage-Regel

Kalibrierung, Modelltraining und Modellselektion verwenden **keine** OOD-/Holdout-Ergebnisse.
Die neuen Stresssets bleiben reine Evaluation. DAPT wird **nicht erneut** trainiert.


In [ ]:

# ============================================================
# 00 – Imports und Konfiguration
# ============================================================
import os, gc, json, math, pickle, random, shutil, zipfile, time, warnings, hashlib
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, confusion_matrix,
    precision_score, recall_score, f1_score
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/phreshphish_hybrid_freeze_extension")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 52, 62, 72, 82]
REPRESENTATIONS = ["BASE", "DAPT", "CONTRASTIVE"]
CLASSIFIERS = ["LOGREG", "XGBOOST", "MLP"]
LABEL_BUDGETS = [0.10, 0.25, 1.00]

TARGET_FPRS = [0.005, 0.010, 0.020]
PRIMARY_TARGET_FPR = 0.005

CALIBRATION_FRACTION = 0.30
CALIBRATION_SPLIT_SEED = 20260808
AP_BALANCE_SEED = 20260809
AP_EQUAL_N_SEED_BASE = 20260820

SYSTEM_LABEL_BUDGET = 0.25
REVIEW_FRACTIONS = [0.05, 0.10, 0.20]

MAX_LENGTH = 256
E2E_EPOCHS = 5
E2E_LR = 2e-5
E2E_WEIGHT_DECAY = 0.01
E2E_WARMUP_RATIO = 0.10
E2E_BATCH_CANDIDATES = [16, 8, 4]
E2E_SCORE_BATCH_SIZE = 64

FREEZE_EXPECTED_SCENARIO_ROWS = {
    "TEMPORAL": 8000,
    "DOMAIN_OOD": 3858,
    "TEMPLATE_OOD": 6832,
    "DOMAIN_TEMPLATE_OOD": 3708,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False

print({
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "output": str(OUTPUT_ROOT),
    "target_fprs": TARGET_FPRS,
    "system_budget": SYSTEM_LABEL_BUDGET,
})


In [ ]:

# ============================================================
# 01 – Eingaben automatisch finden
# ============================================================

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def all_named(name):
    return list(INPUT_ROOT.rglob(name))

def looks_like_freeze_root(p):
    return (
        p.is_dir()
        and (p / "embeddings").exists()
        and (p / "best_classifier_params.json").exists()
        and (p / "tokens").exists()
    )

# FINAL-FREEZE-Output direkt als Kaggle Dataset?
freeze_candidates = [p for p in INPUT_ROOT.rglob("*") if looks_like_freeze_root(p)]
if looks_like_freeze_root(INPUT_ROOT):
    freeze_candidates.append(INPUT_ROOT)

# Oder als ZIP?
if not freeze_candidates:
    extract_root = Path("/kaggle/working/_final_freeze_extract")
    if extract_root.exists():
        shutil.rmtree(extract_root)
    for z in INPUT_ROOT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(z, "r") as zf:
                names = zf.namelist()
                if (
                    any("embeddings/" in n for n in names)
                    and any("best_classifier_params.json" in n for n in names)
                    and any("tokens/" in n for n in names)
                ):
                    extract_root.mkdir(parents=True, exist_ok=True)
                    zf.extractall(extract_root)
                    break
        except Exception:
            continue
    if looks_like_freeze_root(extract_root):
        freeze_candidates.append(extract_root)
    freeze_candidates += [p for p in extract_root.rglob("*") if looks_like_freeze_root(p)]

if not freeze_candidates:
    raise FileNotFoundError(
        "Vollständiger FINAL-FREEZE-Output fehlt. Benötigt werden mindestens "
        "embeddings/, tokens/ und best_classifier_params.json."
    )

FREEZE_ROOT = sorted(set(freeze_candidates), key=lambda p: len(str(p)))[0]

split_paths = all_named("split_roles_and_holdout_cache_v2_ram_safe.pkl")
dapt_paths = all_named("dapt40k_bundle.pkl")
if not split_paths or not dapt_paths:
    raise FileNotFoundError("ENDGAME: Split-Cache bzw. dapt40k_bundle.pkl fehlt.")

SPLIT_PATH = split_paths[0]
DAPT_BUNDLE_PATH = dapt_paths[0]

# DAPT-40k-Encoder für alle fünf Seeds
DAPT_ENCODERS = {}
for seed in SEEDS:
    candidates = []
    for p in INPUT_ROOT.rglob(f"seed_{seed}"):
        s = str(p).lower().replace("\\", "/")
        has_model = (
            (p / "model.safetensors").exists()
            or (p / "pytorch_model.bin").exists()
        )
        if (
            p.is_dir()
            and has_model
            and (p / "config.json").exists()
            and "dapt_encoders" in s
            and "40k" in s
        ):
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError(f"DAPT-40k-Encoder für Seed {seed} nicht gefunden.")
    DAPT_ENCODERS[seed] = sorted(candidates, key=lambda p: len(str(p)))[0]

# Lokales RoBERTa-base für T0
base_candidates = []
for p in INPUT_ROOT.rglob("roberta-base"):
    if not p.is_dir():
        continue
    has_model = (p / "model.safetensors").exists() or (p / "pytorch_model.bin").exists()
    if has_model and (p / "config.json").exists():
        base_candidates.append(p)

if not base_candidates:
    raise FileNotFoundError(
        "Lokales roberta-base-Verzeichnis nicht gefunden. "
        "Bitte das bisherige Offline-RoBERTa-Dataset anhängen."
    )
BASE_MODEL_DIR = sorted(base_candidates, key=lambda p: len(str(p)))[0]

print(json.dumps({
    "freeze_root": str(FREEZE_ROOT),
    "split_cache": str(SPLIT_PATH),
    "dapt_bundle": str(DAPT_BUNDLE_PATH),
    "base_model": str(BASE_MODEL_DIR),
    "dapt_encoders": {str(k): str(v) for k, v in DAPT_ENCODERS.items()},
}, indent=2))


In [ ]:

# ============================================================
# 02 – Datenrollen, 40k-aware Stressszenarien und AP-Testsets
# ============================================================

split_payload = load_pickle(SPLIT_PATH)
dapt_payload = load_pickle(DAPT_BUNDLE_PATH)

def first_existing(obj, keys):
    if not isinstance(obj, dict):
        return None
    for k in keys:
        if k in obj:
            return obj[k]
    return None

train_df = first_existing(
    split_payload, ["train_df", "train", "supervised_train", "downstream_train"]
)
val_df = first_existing(
    split_payload, ["val_df", "validation_df", "validation", "val"]
)
holdout_df = first_existing(
    split_payload, ["final_holdout_clean", "final_holdout", "holdout_df", "holdout"]
)
pretrain_df = first_existing(
    dapt_payload, ["pretrain_large_df", "frame", "pretrain_df", "pretrain"]
)
dapt_holdout = first_existing(dapt_payload, ["final_holdout_clean"])

for name, obj in [
    ("train", train_df), ("validation", val_df),
    ("holdout", holdout_df), ("pretrain40k", pretrain_df)
]:
    if not isinstance(obj, pd.DataFrame):
        raise TypeError(f"{name}: kein DataFrame gefunden.")

train_df = train_df.reset_index(drop=True).copy()
val_df = val_df.reset_index(drop=True).copy()
holdout_df = holdout_df.reset_index(drop=True).copy()
pretrain_df = pretrain_df.reset_index(drop=True).copy()

# 40k-aware Holdout-Flags aus dem DAPT-Bundle priorisieren.
if isinstance(dapt_holdout, pd.DataFrame):
    dapt_holdout = dapt_holdout.reset_index(drop=True)
    if dapt_holdout["sha256"].astype(str).duplicated().any():
        raise RuntimeError("DAPT-Holdout-SHA256 ist nicht eindeutig.")
    lookup = dapt_holdout.copy()
    lookup.index = lookup["sha256"].astype(str)
    hs = holdout_df["sha256"].astype(str)
    for c in [
        "near_duplicate_to_development",
        "min_simhash_distance_to_development",
        "template_seen_in_development",
    ]:
        if c in lookup.columns:
            mapped = hs.map(lookup[c])
            if mapped.isna().any():
                raise RuntimeError(f"40k-aware Holdout-Flag {c} konnte nicht vollständig gemappt werden.")
            holdout_df[c] = mapped.to_numpy()

# Exakt dieselbe 30/70-Aufteilung wie im FINAL FREEZE.
cal_idx, iid_idx = train_test_split(
    np.arange(len(val_df)),
    test_size=1.0 - CALIBRATION_FRACTION,
    random_state=CALIBRATION_SPLIT_SEED,
    stratify=val_df["label"].to_numpy(),
)
calibration_df = val_df.iloc[np.sort(cal_idx)].reset_index(drop=True)
iid_df = val_df.iloc[np.sort(iid_idx)].reset_index(drop=True)

# 40k-aware Development-Identitäten
development_domains = set()
development_templates = set()
for frame in [pretrain_df, train_df, val_df]:
    development_domains.update(frame["domain"].fillna("").astype(str).tolist())
    development_templates.update(frame["template_hash"].fillna("").astype(str).tolist())
development_domains.discard("")
development_templates.discard("")

h_domain = holdout_df["domain"].fillna("").astype(str)
h_template = holdout_df["template_hash"].fillna("").astype(str)
domain_seen = h_domain.isin(development_domains).to_numpy()
template_seen = h_template.isin(development_templates).to_numpy()

if "near_duplicate_to_development" not in holdout_df.columns:
    raise RuntimeError(
        "near_duplicate_to_development fehlt. Template-OOD wird nicht mit einer "
        "schwächeren Definition rekonstruiert."
    )
near_dup = holdout_df["near_duplicate_to_development"].fillna(False).astype(bool).to_numpy()

def balanced_available_indices(frame, mask, seed):
    sub = frame.loc[np.asarray(mask)].copy()
    n0 = int((sub["label"] == 0).sum())
    n1 = int((sub["label"] == 1).sum())
    n_each = min(n0, n1)
    if n_each <= 0:
        raise RuntimeError(f"Szenario nicht balancierbar: {n0=} {n1=}")
    p0 = sub[sub["label"].eq(0)].sample(n=n_each, random_state=seed)
    p1 = sub[sub["label"].eq(1)].sample(n=n_each, random_state=seed + 1)
    return np.sort(pd.concat([p0, p1]).index.to_numpy(dtype=np.int32))

domain_new_mask = (~domain_seen) & h_domain.ne("").to_numpy()
template_ood_mask = (~template_seen) & (~near_dup) & h_template.ne("").to_numpy()
domain_template_mask = domain_new_mask & template_ood_mask

stress_indices = {
    "TEMPORAL": np.arange(len(holdout_df), dtype=np.int32),
    "DOMAIN_OOD": balanced_available_indices(holdout_df, domain_new_mask, 108),
    "TEMPLATE_OOD": balanced_available_indices(holdout_df, template_ood_mask, 109),
    "DOMAIN_TEMPLATE_OOD": balanced_available_indices(holdout_df, domain_template_mask, 110),
}

# Harte Regression-Guards des FINAL FREEZE.
for name, expected_n in FREEZE_EXPECTED_SCENARIO_ROWS.items():
    idx = stress_indices[name]
    labels = holdout_df.iloc[idx]["label"].to_numpy(dtype=int)
    actual_n = len(idx)
    if actual_n != expected_n:
        raise RuntimeError(f"{name}: {actual_n} statt eingefroren {expected_n}.")
    if int((labels == 0).sum()) != int((labels == 1).sum()):
        raise RuntimeError(f"{name}: nicht 50/50 balanciert.")

# ------------------------------------------------------------
# AP-Korrektur
# ------------------------------------------------------------
# IID_ORIGINAL: für operative Kennzahlen unverändert beibehalten.
iid_y = iid_df["label"].to_numpy(dtype=int)
iid_pos = np.flatnonzero(iid_y == 1)
iid_neg = np.flatnonzero(iid_y == 0)
n_each_iid = min(len(iid_pos), len(iid_neg))

rng = np.random.default_rng(AP_BALANCE_SEED)
iid_neg_sample = rng.choice(iid_neg, size=n_each_iid, replace=False)
iid_pos_sample = rng.choice(iid_pos, size=n_each_iid, replace=False)
IID_BALANCED_IDX = np.sort(
    np.concatenate([iid_neg_sample, iid_pos_sample]).astype(np.int32)
)

# Primary AP comparison:
# gleiche Prävalenz (50/50), OOD-Daten vollständig behalten.
SCENARIOS = {
    "IID_ORIGINAL": ("iid", np.arange(len(iid_df), dtype=np.int32)),
    "IID_BALANCED": ("iid", IID_BALANCED_IDX),
    "TEMPORAL": ("holdout", stress_indices["TEMPORAL"]),
    "DOMAIN_OOD": ("holdout", stress_indices["DOMAIN_OOD"]),
    "TEMPLATE_OOD": ("holdout", stress_indices["TEMPLATE_OOD"]),
    "DOMAIN_TEMPLATE_OOD": ("holdout", stress_indices["DOMAIN_TEMPLATE_OOD"]),
}
AP_COMPARABLE_SCENARIOS = [
    "IID_BALANCED", "TEMPORAL", "DOMAIN_OOD",
    "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
]

# Zusätzliche Equal-N-Sensitivität:
# exakt n_each_iid Benign + n_each_iid Phishing in JEDEM AP-Szenario.
AP_EQUAL_N_INDICES = {"IID_BALANCED": IID_BALANCED_IDX}
for offset, name in enumerate([
    "TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
]):
    full_idx = stress_indices[name]
    y = holdout_df.iloc[full_idx]["label"].to_numpy(dtype=int)
    local0 = np.flatnonzero(y == 0)
    local1 = np.flatnonzero(y == 1)
    if min(len(local0), len(local1)) < n_each_iid:
        raise RuntimeError(f"{name}: zu klein für Equal-N AP mit {n_each_iid} je Klasse.")
    rr = np.random.default_rng(AP_EQUAL_N_SEED_BASE + offset)
    chosen_local = np.concatenate([
        rr.choice(local0, n_each_iid, replace=False),
        rr.choice(local1, n_each_iid, replace=False),
    ])
    AP_EQUAL_N_INDICES[name] = np.sort(full_idx[chosen_local].astype(np.int32))

# Audit
audit_rows = []
for name, (source, idx) in SCENARIOS.items():
    frame = iid_df.iloc[idx] if source == "iid" else holdout_df.iloc[idx]
    n0 = int((frame["label"] == 0).sum())
    n1 = int((frame["label"] == 1).sum())
    audit_rows.append({
        "set_family": "PRIMARY",
        "scenario": name,
        "source": source,
        "n": len(frame),
        "benign": n0,
        "phish": n1,
        "prevalence_phish": n1 / len(frame),
    })

for name, idx in AP_EQUAL_N_INDICES.items():
    if name == "IID_BALANCED":
        frame = iid_df.iloc[idx]
        source = "iid"
    else:
        frame = holdout_df.iloc[idx]
        source = "holdout"
    n0 = int((frame["label"] == 0).sum())
    n1 = int((frame["label"] == 1).sum())
    audit_rows.append({
        "set_family": "AP_EQUAL_N",
        "scenario": name,
        "source": source,
        "n": len(frame),
        "benign": n0,
        "phish": n1,
        "prevalence_phish": n1 / len(frame),
    })

testset_audit = pd.DataFrame(audit_rows)
testset_audit.to_csv(OUTPUT_ROOT / "testset_prevalence_audit.csv", index=False)
display(testset_audit)

print({
    "train": len(train_df),
    "calibration": len(calibration_df),
    "iid_original": len(iid_df),
    "iid_balanced": len(IID_BALANCED_IDX),
    "iid_balanced_each_class": n_each_iid,
    "holdout": len(holdout_df),
    "dapt40k": len(pretrain_df),
})


In [ ]:

# ============================================================
# 03 – FINAL-FREEZE Embeddings, Token-Caches und Labelbudgets
# ============================================================

with open(FREEZE_ROOT / "best_classifier_params.json", "r", encoding="utf-8") as f:
    BEST_PARAMS = json.load(f)
if "hidden_layer_sizes" in BEST_PARAMS["MLP"]:
    BEST_PARAMS["MLP"]["hidden_layer_sizes"] = tuple(
        BEST_PARAMS["MLP"]["hidden_layer_sizes"]
    )

EMBED_ROOT = FREEZE_ROOT / "embeddings"
TOKEN_ROOT = FREEZE_ROOT / "tokens"

def emb_path(rep, seed, split):
    seed_key = "shared" if rep == "BASE" else f"seed_{seed}"
    return EMBED_ROOT / rep.lower() / seed_key / f"{split}.npy"

def load_emb(rep, seed, split):
    effective_seed = SEEDS[0] if rep == "BASE" else seed
    p = emb_path(rep, effective_seed, split)
    if not p.exists():
        raise FileNotFoundError(p)
    return np.asarray(np.load(p, mmap_mode="r"), dtype=np.float32)

def load_token_split(folder):
    root = TOKEN_ROOT / folder
    ids = root / "input_ids.npy"
    mask = root / "attention_mask.npy"
    if not ids.exists() or not mask.exists():
        raise FileNotFoundError(f"Token-Cache fehlt: {root}")
    return {
        "input_ids": np.load(ids, mmap_mode="r"),
        "attention_mask": np.load(mask, mmap_mode="r"),
    }

TOKENS = {
    "train": load_token_split("train_4k"),
    "calibration": load_token_split("calibration"),
    "iid": load_token_split("iid_test"),
    "holdout": load_token_split("holdout_8k"),
}

expected_token_n = {
    "train": len(train_df),
    "calibration": len(calibration_df),
    "iid": len(iid_df),
    "holdout": len(holdout_df),
}
for name, arrays in TOKENS.items():
    if arrays["input_ids"].shape[0] != expected_token_n[name]:
        raise RuntimeError(
            f"Token-Cache {name}: {arrays['input_ids'].shape[0]} "
            f"statt {expected_token_n[name]}."
        )
    if arrays["input_ids"].shape[1] != MAX_LENGTH:
        raise RuntimeError(f"Token-Cache {name}: max_length ist nicht {MAX_LENGTH}.")

y_train = train_df["label"].to_numpy(dtype=int)

def nested_budget_indices(y, seed):
    rng = np.random.default_rng(seed)
    class_orders = {}
    for c in [0, 1]:
        idx = np.flatnonzero(y == c).copy()
        rng.shuffle(idx)
        class_orders[c] = idx
    result = {}
    for frac in LABEL_BUDGETS:
        parts = []
        for c in [0, 1]:
            n = len(class_orders[c]) if frac == 1.0 else max(
                1, int(round(len(class_orders[c]) * frac))
            )
            parts.append(class_orders[c][:n])
        result[frac] = np.sort(np.concatenate(parts)).astype(np.int32)
    assert set(result[0.10]).issubset(set(result[0.25]))
    assert set(result[0.25]).issubset(set(result[1.00]))
    return result

BUDGET_INDICES = {seed: nested_budget_indices(y_train, seed) for seed in SEEDS}

budget_rows = []
for seed, d in BUDGET_INDICES.items():
    for frac, idx in d.items():
        budget_rows.append({
            "seed": seed,
            "budget": frac,
            "n": len(idx),
            "benign": int((y_train[idx] == 0).sum()),
            "phish": int((y_train[idx] == 1).sum()),
        })
pd.DataFrame(budget_rows).to_csv(OUTPUT_ROOT / "label_budget_audit.csv", index=False)
display(pd.DataFrame(budget_rows))


In [ ]:

# ============================================================
# 04 – Gemeinsame Klassifikatoren, FPR-Schwellen und Metriken
# ============================================================

def build_classifier(kind, params, seed):
    if kind == "LOGREG":
        return Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(
                C=params["C"],
                max_iter=2500,
                solver="lbfgs",
                random_state=seed,
            )),
        ])
    if kind == "XGBOOST":
        return XGBClassifier(
            **params,
            subsample=0.9,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=2,
            random_state=seed,
        )
    if kind == "MLP":
        return Pipeline([
            ("scale", StandardScaler()),
            ("clf", MLPClassifier(
                **params,
                activation="relu",
                solver="adam",
                batch_size=128,
                learning_rate_init=1e-3,
                max_iter=200,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=12,
                random_state=seed,
            )),
        ])
    raise KeyError(kind)

def threshold_for_target_fpr(y_true, score, target_fpr):
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(score, dtype=float)
    neg = np.sort(s[y == 0])[::-1]
    if len(neg) == 0:
        raise ValueError("Keine Negativklasse für FPR-Kalibrierung.")
    allowed = int(math.floor(target_fpr * len(neg) + 1e-12))
    if allowed <= 0:
        return float(np.nextafter(neg[0], np.inf))
    if allowed >= len(neg):
        return float(-np.inf)
    return float(np.nextafter(neg[allowed], np.inf))

def metric_row(y, score, threshold):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(y, score)),
        "roc_auc": float(roc_auc_score(y, score)) if len(np.unique(y)) == 2 else np.nan,
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "empirical_fpr": float(fp / max(fp + tn, 1)),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "prevalence_phish": float((y == 1).mean()),
    }

def calibration_operating_points(ycal, cal_score):
    out = {}
    for target in TARGET_FPRS:
        thr = threshold_for_target_fpr(ycal, cal_score, target)
        m = metric_row(ycal, cal_score, thr)
        out[target] = {
            "threshold": thr,
            "calibration_empirical_fpr": m["empirical_fpr"],
            "calibration_recall": m["recall"],
        }
    return out

def scenario_score_view(iid_score, holdout_score, scenario):
    source, idx = SCENARIOS[scenario]
    if source == "iid":
        y = iid_df.iloc[idx]["label"].to_numpy(dtype=int)
        return y, np.asarray(iid_score)[idx]
    y = holdout_df.iloc[idx]["label"].to_numpy(dtype=int)
    return y, np.asarray(holdout_score)[idx]

def equal_n_score_view(iid_score, holdout_score, scenario):
    idx = AP_EQUAL_N_INDICES[scenario]
    if scenario == "IID_BALANCED":
        y = iid_df.iloc[idx]["label"].to_numpy(dtype=int)
        return y, np.asarray(iid_score)[idx]
    y = holdout_df.iloc[idx]["label"].to_numpy(dtype=int)
    return y, np.asarray(holdout_score)[idx]


In [ ]:

# ============================================================
# 05 – FINAL FREEZE korrigiert neu auswerten
#      3 Repräsentationen × 3 Classifier × 5 Seeds × 3 Budgets
# ============================================================

FREEZE_SCORE_ROOT = OUTPUT_ROOT / "freeze_scores"
FREEZE_SCORE_ROOT.mkdir(exist_ok=True)

def freeze_score_path(rep, kind, seed, frac):
    b = str(frac).replace(".", "p")
    return FREEZE_SCORE_ROOT / f"{rep}_{kind}_seed{seed}_budget{b}.npz"

freeze_rows = []

for seed in SEEDS:
    for frac in LABEL_BUDGETS:
        train_idx = BUDGET_INDICES[seed][frac]
        yb = y_train[train_idx]
        for rep in REPRESENTATIONS:
            Xtrain = load_emb(rep, seed, "train")[train_idx]
            Xcal = load_emb(rep, seed, "calibration")
            Xiid = load_emb(rep, seed, "iid")
            Xhold = load_emb(rep, seed, "holdout")
            ycal = calibration_df["label"].to_numpy(dtype=int)

            for kind in CLASSIFIERS:
                score_file = freeze_score_path(rep, kind, seed, frac)

                if score_file.exists():
                    dat = np.load(score_file)
                    cal_score = dat["cal"].astype(np.float64)
                    iid_score = dat["iid"].astype(np.float64)
                    hold_score = dat["holdout"].astype(np.float64)
                    fit_seconds = float(dat["fit_seconds"][0]) if "fit_seconds" in dat.files else np.nan
                else:
                    t0 = time.perf_counter()
                    model = build_classifier(kind, BEST_PARAMS[kind], seed)
                    model.fit(Xtrain, yb)
                    fit_seconds = time.perf_counter() - t0
                    cal_score = model.predict_proba(Xcal)[:, 1]
                    iid_score = model.predict_proba(Xiid)[:, 1]
                    hold_score = model.predict_proba(Xhold)[:, 1]
                    np.savez_compressed(
                        score_file,
                        cal=cal_score.astype(np.float32),
                        iid=iid_score.astype(np.float32),
                        holdout=hold_score.astype(np.float32),
                        fit_seconds=np.asarray([fit_seconds], dtype=np.float64),
                    )
                    del model

                ops = calibration_operating_points(ycal, cal_score)

                for target in TARGET_FPRS:
                    op = ops[target]
                    for scenario in SCENARIOS:
                        yev, sev = scenario_score_view(iid_score, hold_score, scenario)
                        freeze_rows.append({
                            "representation": rep,
                            "classifier": kind,
                            "seed": seed,
                            "label_budget": float(frac),
                            "n_train": int(len(train_idx)),
                            "scenario": scenario,
                            "n_test": int(len(yev)),
                            "target_fpr": float(target),
                            "threshold": float(op["threshold"]),
                            "calibration_empirical_fpr": float(op["calibration_empirical_fpr"]),
                            "calibration_recall": float(op["calibration_recall"]),
                            "fit_seconds": fit_seconds,
                            **metric_row(yev, sev, op["threshold"]),
                        })

                print({
                    "freeze_corrected": [rep, kind, seed, frac],
                    "rows": len(freeze_rows),
                })

            del Xtrain, Xcal, Xiid, Xhold
            gc.collect()

freeze_results = pd.DataFrame(freeze_rows)
freeze_results.to_csv(OUTPUT_ROOT / "freeze_corrected_results.csv", index=False)

# Zusammenfassung: FPR ist ausdrücklich Teil jeder Tabellenzeile.
freeze_summary = (
    freeze_results
    .groupby(
        ["representation", "classifier", "label_budget", "scenario", "target_fpr"],
        as_index=False
    )
    .agg(
        n_test=("n_test", "first"),
        prevalence_phish=("prevalence_phish", "first"),
        average_precision_mean=("average_precision", "mean"),
        average_precision_std=("average_precision", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        calibration_empirical_fpr_mean=("calibration_empirical_fpr", "mean"),
        empirical_fpr_mean=("empirical_fpr", "mean"),
        empirical_fpr_std=("empirical_fpr", "std"),
        fp_mean=("fp", "mean"),
        fn_mean=("fn", "mean"),
    )
)
freeze_summary.to_csv(OUTPUT_ROOT / "freeze_corrected_summary.csv", index=False)
display(freeze_summary.head(30))


In [ ]:

# ============================================================
# 06 – AP-Shift korrigiert:
#      primär gleiche Prävalenz; zusätzlich Equal-N Sensitivität
# ============================================================

ap_shift_rows = []

key_cols = ["representation", "classifier", "seed", "label_budget", "target_fpr"]

for keys, g in freeze_results.groupby(key_cols):
    iid_row = g[g["scenario"] == "IID_BALANCED"]
    if len(iid_row) != 1:
        raise RuntimeError(f"IID_BALANCED fehlt/dupliziert für {keys}")
    iid_row = iid_row.iloc[0]

    for scenario in ["TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"]:
        ood = g[g["scenario"] == scenario]
        if len(ood) != 1:
            raise RuntimeError(f"{scenario} fehlt/dupliziert für {keys}")
        ood = ood.iloc[0]
        ap_shift_rows.append({
            "representation": keys[0],
            "classifier": keys[1],
            "seed": keys[2],
            "label_budget": keys[3],
            "target_fpr": keys[4],
            "comparison": f"IID_BALANCED->{scenario}",
            "iid_n": int(iid_row["n_test"]),
            "ood_n": int(ood["n_test"]),
            "iid_prevalence": float(iid_row["prevalence_phish"]),
            "ood_prevalence": float(ood["prevalence_phish"]),
            "iid_ap": float(iid_row["average_precision"]),
            "ood_ap": float(ood["average_precision"]),
            "ap_delta_ood_minus_iid": float(
                ood["average_precision"] - iid_row["average_precision"]
            ),
            "iid_empirical_fpr": float(iid_row["empirical_fpr"]),
            "ood_empirical_fpr": float(ood["empirical_fpr"]),
            "empirical_fpr_delta": float(
                ood["empirical_fpr"] - iid_row["empirical_fpr"]
            ),
            "iid_recall": float(iid_row["recall"]),
            "ood_recall": float(ood["recall"]),
            "recall_delta": float(ood["recall"] - iid_row["recall"]),
        })

ap_shift = pd.DataFrame(ap_shift_rows)
ap_shift.to_csv(OUTPUT_ROOT / "ap_shift_corrected_prevalence_matched.csv", index=False)

# Equal-N-Sensitivität wird aus denselben gespeicherten Scores berechnet.
equal_n_rows = []
for seed in SEEDS:
    for frac in LABEL_BUDGETS:
        for rep in REPRESENTATIONS:
            for kind in CLASSIFIERS:
                dat = np.load(freeze_score_path(rep, kind, seed, frac))
                cal_score = dat["cal"]
                iid_score = dat["iid"]
                hold_score = dat["holdout"]
                ycal = calibration_df["label"].to_numpy(dtype=int)
                ops = calibration_operating_points(ycal, cal_score)

                for target in TARGET_FPRS:
                    op = ops[target]
                    for scenario in [
                        "IID_BALANCED", "TEMPORAL", "DOMAIN_OOD",
                        "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
                    ]:
                        yev, sev = equal_n_score_view(iid_score, hold_score, scenario)
                        equal_n_rows.append({
                            "representation": rep,
                            "classifier": kind,
                            "seed": seed,
                            "label_budget": float(frac),
                            "scenario": scenario,
                            "n_test": len(yev),
                            "target_fpr": target,
                            "threshold": op["threshold"],
                            "calibration_empirical_fpr": op["calibration_empirical_fpr"],
                            **metric_row(yev, sev, op["threshold"]),
                        })

equal_n_results = pd.DataFrame(equal_n_rows)
equal_n_results.to_csv(OUTPUT_ROOT / "ap_equal_n_sensitivity_results.csv", index=False)

equal_n_shift_rows = []
for keys, g in equal_n_results.groupby(key_cols):
    iid = g[g["scenario"] == "IID_BALANCED"].iloc[0]
    for scenario in ["TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"]:
        ood = g[g["scenario"] == scenario].iloc[0]
        equal_n_shift_rows.append({
            "representation": keys[0],
            "classifier": keys[1],
            "seed": keys[2],
            "label_budget": keys[3],
            "target_fpr": keys[4],
            "comparison": f"IID_BALANCED->{scenario}",
            "n_each_test": int(iid["n_test"]),
            "iid_ap": float(iid["average_precision"]),
            "ood_ap": float(ood["average_precision"]),
            "ap_delta_ood_minus_iid": float(ood["average_precision"] - iid["average_precision"]),
            "iid_empirical_fpr": float(iid["empirical_fpr"]),
            "ood_empirical_fpr": float(ood["empirical_fpr"]),
            "empirical_fpr_delta": float(ood["empirical_fpr"] - iid["empirical_fpr"]),
        })

equal_n_shift = pd.DataFrame(equal_n_shift_rows)
equal_n_shift.to_csv(OUTPUT_ROOT / "ap_shift_equal_n_sensitivity.csv", index=False)

print({
    "primary_ap_shift_rows": len(ap_shift),
    "equal_n_rows": len(equal_n_results),
    "equal_n_shift_rows": len(equal_n_shift),
})



## Systemerweiterung bei 25 % Labels

Die folgende Stufe bringt die **starke klassische Struktur-Baseline** und die
**End-to-End-Transformer** aus der früheren Versuchslinie zurück, verwendet aber die
**jetzt eingefrorenen 40k-aware Testdefinitionen**.

Damit werden keine alten, methodisch anders definierten OOD-Zahlen mit dem FINAL FREEZE
vermischt.

- `B0_STRUCT_XGB` wird auf den expliziten Strukturmerkmalen trainiert.
- Seine XGBoost-Parameter werden **nur innerhalb des 25-%-Trainingssubsets** per 3-fold CV gewählt.
- `T0_E2E` und `DAPT_E2E` erhalten exakt dieselben Labels, dieselben Tokens und denselben
  Fine-Tuning-Ablauf.
- DAPT-40k wird **nicht neu vortrainiert**; die vorhandenen fünf DAPT-Encoder werden geladen.
- Für `BASE_EMB_MLP`, `DAPT_EMB_MLP` und `CONTRASTIVE_EMB_MLP` werden die Scores aus
  der korrigierten Freeze-Auswertung wiederverwendet.


In [ ]:

# ============================================================
# 07 – Explizite Strukturmerkmale für B0
# ============================================================

def normalize_feature_obj(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            obj = json.loads(x)
            return obj if isinstance(obj, dict) else {}
        except Exception:
            return {}
    return {}

if "features" not in train_df.columns:
    raise KeyError(
        "Spalte 'features' fehlt. B0_STRUCT_XGB benötigt die expliziten "
        "URL-/HTML-Strukturmerkmale aus dem Split-Cache."
    )

train_feature_dicts = [normalize_feature_obj(x) for x in train_df["features"]]
feature_frame_train = pd.json_normalize(train_feature_dicts, sep="__")

# Nur numerisch interpretierbare Spalten zulassen.
numeric_cols = []
for c in feature_frame_train.columns:
    converted = pd.to_numeric(feature_frame_train[c], errors="coerce")
    if converted.notna().any():
        numeric_cols.append(c)

if not numeric_cols:
    raise RuntimeError("Keine numerischen Strukturmerkmale in train_df['features'] gefunden.")

FEATURE_COLUMNS = sorted(numeric_cols)

def feature_matrix(frame):
    tmp = pd.json_normalize(
        [normalize_feature_obj(x) for x in frame["features"]],
        sep="__"
    )
    tmp = tmp.reindex(columns=FEATURE_COLUMNS)
    for c in FEATURE_COLUMNS:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
    return tmp.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float32)

X_STRUCT = {
    "train": feature_matrix(train_df),
    "calibration": feature_matrix(calibration_df),
    "iid": feature_matrix(iid_df),
    "holdout": feature_matrix(holdout_df),
}

pd.DataFrame({"feature": FEATURE_COLUMNS}).to_csv(
    OUTPUT_ROOT / "b0_structural_feature_columns.csv", index=False
)

print({
    "b0_n_features": len(FEATURE_COLUMNS),
    "train_shape": X_STRUCT["train"].shape,
})


In [ ]:

# ============================================================
# 08 – B0_STRUCT_XGB: train-only CV bei 25 % Labels
# ============================================================

B0_CANDIDATES = [
    {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.05, "min_child_weight": 1},
    {"n_estimators": 500, "max_depth": 3, "learning_rate": 0.05, "min_child_weight": 1},
    {"n_estimators": 400, "max_depth": 4, "learning_rate": 0.10, "min_child_weight": 2},
    {"n_estimators": 400, "max_depth": 5, "learning_rate": 0.05, "min_child_weight": 1},
]

def build_b0(params, seed):
    return XGBClassifier(
        **params,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=2,
        random_state=seed,
    )

SYSTEM_SCORE_ROOT = OUTPUT_ROOT / "system_scores"
SYSTEM_SCORE_ROOT.mkdir(exist_ok=True)

def system_score_path(model_name, seed):
    return SYSTEM_SCORE_ROOT / f"{model_name}_seed{seed}.npz"

b0_tuning_rows = []

for seed in SEEDS:
    idx = BUDGET_INDICES[seed][SYSTEM_LABEL_BUDGET]
    X = X_STRUCT["train"][idx]
    y = y_train[idx]

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    best = None
    for ci, params in enumerate(B0_CANDIDATES):
        vals = []
        for fold, (tr, va) in enumerate(cv.split(X, y)):
            m = build_b0(params, seed + fold)
            m.fit(X[tr], y[tr])
            p = m.predict_proba(X[va])[:, 1]
            vals.append(average_precision_score(y[va], p))
        mean_ap = float(np.mean(vals))
        b0_tuning_rows.append({
            "seed": seed,
            "candidate": ci,
            "params": json.dumps(params, sort_keys=True),
            "cv_ap": mean_ap,
        })
        if best is None or mean_ap > best[0]:
            best = (mean_ap, params)

    score_file = system_score_path("B0_STRUCT_XGB", seed)
    if not score_file.exists():
        t0 = time.perf_counter()
        model = build_b0(best[1], seed)
        model.fit(X, y)
        fit_seconds = time.perf_counter() - t0
        np.savez_compressed(
            score_file,
            cal=model.predict_proba(X_STRUCT["calibration"])[:, 1].astype(np.float32),
            iid=model.predict_proba(X_STRUCT["iid"])[:, 1].astype(np.float32),
            holdout=model.predict_proba(X_STRUCT["holdout"])[:, 1].astype(np.float32),
            fit_seconds=np.asarray([fit_seconds]),
            params_json=np.asarray([json.dumps(best[1], sort_keys=True)]),
        )
        del model

    print({
        "b0_seed": seed,
        "n_labels": len(idx),
        "best_train_cv_ap": round(best[0], 6),
        "params": best[1],
    })

b0_tuning_df = pd.DataFrame(b0_tuning_rows)
b0_tuning_df.to_csv(OUTPUT_ROOT / "b0_train_only_tuning.csv", index=False)


In [ ]:

# ============================================================
# 09 – T0_E2E und DAPT_E2E
#      gleiche Tokens, Labels und Fine-Tuning-Prozedur
# ============================================================

class TokenSubsetDataset(Dataset):
    def __init__(self, arrays, labels, indices):
        self.ids = arrays["input_ids"]
        self.mask = arrays["attention_mask"]
        self.labels = np.asarray(labels, dtype=np.int64)
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = int(self.indices[i])
        return (
            torch.tensor(self.ids[j], dtype=torch.long),
            torch.tensor(self.mask[j], dtype=torch.long),
            torch.tensor(self.labels[j], dtype=torch.long),
        )

class TokenAllDataset(Dataset):
    def __init__(self, arrays):
        self.ids = arrays["input_ids"]
        self.mask = arrays["attention_mask"]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return (
            torch.tensor(self.ids[i], dtype=torch.long),
            torch.tensor(self.mask[i], dtype=torch.long),
        )

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def score_e2e_model(model, arrays, batch_size=E2E_SCORE_BATCH_SIZE):
    ds = TokenAllDataset(arrays)
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=torch.cuda.is_available()
    )
    model.eval()
    scores = []
    use_amp = torch.cuda.is_available()
    for ids, mask in loader:
        ids = ids.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(
            device_type="cuda", dtype=torch.float16, enabled=use_amp
        ):
            logits = model(input_ids=ids, attention_mask=mask).logits
        scores.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(scores).astype(np.float32)

def train_one_e2e(init_dir, seed, train_idx, model_name):
    score_file = system_score_path(model_name, seed)
    if score_file.exists():
        print({"e2e_reuse": model_name, "seed": seed})
        return

    y = y_train
    last_error = None

    for batch_size in E2E_BATCH_CANDIDATES:
        try:
            set_all_seeds(seed)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            model = AutoModelForSequenceClassification.from_pretrained(
                str(init_dir),
                num_labels=2,
                local_files_only=True,
                ignore_mismatched_sizes=True,
            )
            model.to(DEVICE)

            ds = TokenSubsetDataset(TOKENS["train"], y, train_idx)
            generator = torch.Generator()
            generator.manual_seed(seed)
            loader = DataLoader(
                ds,
                batch_size=batch_size,
                shuffle=True,
                generator=generator,
                num_workers=2,
                pin_memory=torch.cuda.is_available(),
            )

            # OOM-Fallback hält die effektive Batchgröße ungefähr bei 16.
            grad_accum = max(1, 16 // batch_size)
            updates_per_epoch = math.ceil(len(loader) / grad_accum)
            total_updates = updates_per_epoch * E2E_EPOCHS
            warmup_steps = int(round(total_updates * E2E_WARMUP_RATIO))

            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=E2E_LR,
                weight_decay=E2E_WEIGHT_DECAY,
            )
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_updates,
            )

            use_amp = torch.cuda.is_available()
            scaler = torch.amp.GradScaler("cuda") if use_amp else None

            history = []
            started = time.perf_counter()

            for epoch in range(E2E_EPOCHS):
                model.train()
                optimizer.zero_grad(set_to_none=True)
                losses = []

                for step, (ids, mask, labels) in enumerate(loader):
                    ids = ids.to(DEVICE, non_blocking=True)
                    mask = mask.to(DEVICE, non_blocking=True)
                    labels = labels.to(DEVICE, non_blocking=True)

                    with torch.amp.autocast(
                        device_type="cuda", dtype=torch.float16, enabled=use_amp
                    ):
                        out = model(
                            input_ids=ids,
                            attention_mask=mask,
                            labels=labels,
                        )
                        loss = out.loss / grad_accum

                    if scaler is not None:
                        scaler.scale(loss).backward()
                    else:
                        loss.backward()

                    losses.append(float(loss.detach().cpu()) * grad_accum)

                    should_step = ((step + 1) % grad_accum == 0) or (step + 1 == len(loader))
                    if should_step:
                        if scaler is not None:
                            scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        if scaler is not None:
                            scaler.step(optimizer)
                            scaler.update()
                        else:
                            optimizer.step()
                        scheduler.step()
                        optimizer.zero_grad(set_to_none=True)

                history.append({
                    "model": model_name,
                    "seed": seed,
                    "epoch": epoch + 1,
                    "train_loss": float(np.mean(losses)),
                    "batch_size": batch_size,
                    "gradient_accumulation": grad_accum,
                })
                print(history[-1])

            fit_seconds = time.perf_counter() - started

            cal_score = score_e2e_model(model, TOKENS["calibration"])
            iid_score = score_e2e_model(model, TOKENS["iid"])
            hold_score = score_e2e_model(model, TOKENS["holdout"])

            np.savez_compressed(
                score_file,
                cal=cal_score,
                iid=iid_score,
                holdout=hold_score,
                fit_seconds=np.asarray([fit_seconds]),
                batch_size=np.asarray([batch_size]),
                grad_accum=np.asarray([grad_accum]),
            )

            hist_path = OUTPUT_ROOT / f"{model_name}_training_history_seed{seed}.csv"
            pd.DataFrame(history).to_csv(hist_path, index=False)

            del model, optimizer, scheduler, loader, ds
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return

        except RuntimeError as exc:
            last_error = exc
            if "out of memory" not in str(exc).lower():
                raise
            print({
                "oom_retry": model_name,
                "seed": seed,
                "failed_batch_size": batch_size,
            })
            try:
                del model
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    raise RuntimeError(
        f"{model_name} Seed {seed}: alle Batchgrößen fehlgeschlagen."
    ) from last_error

for seed in SEEDS:
    train_idx = BUDGET_INDICES[seed][SYSTEM_LABEL_BUDGET]

    train_one_e2e(
        BASE_MODEL_DIR,
        seed,
        train_idx,
        "T0_E2E",
    )
    train_one_e2e(
        DAPT_ENCODERS[seed],
        seed,
        train_idx,
        "DAPT_E2E",
    )


In [ ]:

# ============================================================
# 10 – Embedding-Systemkandidaten aus dem FINAL FREEZE übernehmen
# ============================================================

EMBED_SYSTEMS = {
    "BASE_EMB_MLP": ("BASE", "MLP"),
    "DAPT_EMB_MLP": ("DAPT", "MLP"),
    "CONTRASTIVE_EMB_MLP": ("CONTRASTIVE", "MLP"),
}

for seed in SEEDS:
    for system_name, (rep, kind) in EMBED_SYSTEMS.items():
        src = freeze_score_path(rep, kind, seed, SYSTEM_LABEL_BUDGET)
        if not src.exists():
            raise FileNotFoundError(src)
        dat = np.load(src)
        dst = system_score_path(system_name, seed)
        if not dst.exists():
            np.savez_compressed(
                dst,
                cal=dat["cal"].astype(np.float32),
                iid=dat["iid"].astype(np.float32),
                holdout=dat["holdout"].astype(np.float32),
                fit_seconds=dat["fit_seconds"] if "fit_seconds" in dat.files else np.asarray([np.nan]),
            )

print({"embedding_system_scores": "ready"})


In [ ]:

# ============================================================
# 11 – Systemvergleich bei 25 % Labels, FPR überall
# ============================================================

SYSTEM_MODELS = [
    "B0_STRUCT_XGB",
    "T0_E2E",
    "DAPT_E2E",
    "BASE_EMB_MLP",
    "DAPT_EMB_MLP",
    "CONTRASTIVE_EMB_MLP",
]

system_rows = []

for seed in SEEDS:
    ycal = calibration_df["label"].to_numpy(dtype=int)

    for model_name in SYSTEM_MODELS:
        dat = np.load(system_score_path(model_name, seed))
        cal_score = dat["cal"]
        iid_score = dat["iid"]
        hold_score = dat["holdout"]
        fit_seconds = float(dat["fit_seconds"][0]) if "fit_seconds" in dat.files else np.nan
        ops = calibration_operating_points(ycal, cal_score)

        for target in TARGET_FPRS:
            op = ops[target]
            for scenario in SCENARIOS:
                yev, sev = scenario_score_view(iid_score, hold_score, scenario)
                system_rows.append({
                    "model": model_name,
                    "seed": seed,
                    "label_budget": SYSTEM_LABEL_BUDGET,
                    "n_train": len(BUDGET_INDICES[seed][SYSTEM_LABEL_BUDGET]),
                    "scenario": scenario,
                    "n_test": len(yev),
                    "target_fpr": target,
                    "threshold": op["threshold"],
                    "calibration_empirical_fpr": op["calibration_empirical_fpr"],
                    "calibration_recall": op["calibration_recall"],
                    "fit_seconds": fit_seconds,
                    **metric_row(yev, sev, op["threshold"]),
                })

system_results = pd.DataFrame(system_rows)
system_results.to_csv(OUTPUT_ROOT / "system_25pct_results.csv", index=False)

system_summary = (
    system_results
    .groupby(["model", "scenario", "target_fpr"], as_index=False)
    .agg(
        n_test=("n_test", "first"),
        prevalence_phish=("prevalence_phish", "first"),
        average_precision_mean=("average_precision", "mean"),
        average_precision_std=("average_precision", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        calibration_empirical_fpr_mean=("calibration_empirical_fpr", "mean"),
        empirical_fpr_mean=("empirical_fpr", "mean"),
        empirical_fpr_std=("empirical_fpr", "std"),
        fp_mean=("fp", "mean"),
        fn_mean=("fn", "mean"),
    )
)
system_summary.to_csv(OUTPUT_ROOT / "system_25pct_summary.csv", index=False)
display(system_summary)


In [ ]:

# ============================================================
# 12 – Korrigierter AP-Shift auf Systemebene
# ============================================================

system_ap_shift_rows = []

for keys, g in system_results.groupby(["model", "seed", "target_fpr"]):
    iid = g[g["scenario"] == "IID_BALANCED"].iloc[0]
    for scenario in ["TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"]:
        ood = g[g["scenario"] == scenario].iloc[0]
        system_ap_shift_rows.append({
            "model": keys[0],
            "seed": keys[1],
            "target_fpr": keys[2],
            "comparison": f"IID_BALANCED->{scenario}",
            "iid_n": int(iid["n_test"]),
            "ood_n": int(ood["n_test"]),
            "iid_prevalence": float(iid["prevalence_phish"]),
            "ood_prevalence": float(ood["prevalence_phish"]),
            "iid_ap": float(iid["average_precision"]),
            "ood_ap": float(ood["average_precision"]),
            "ap_delta_ood_minus_iid": float(ood["average_precision"] - iid["average_precision"]),
            "iid_empirical_fpr": float(iid["empirical_fpr"]),
            "ood_empirical_fpr": float(ood["empirical_fpr"]),
            "empirical_fpr_delta": float(ood["empirical_fpr"] - iid["empirical_fpr"]),
            "iid_recall": float(iid["recall"]),
            "ood_recall": float(ood["recall"]),
            "recall_delta": float(ood["recall"] - iid["recall"]),
        })

system_ap_shift = pd.DataFrame(system_ap_shift_rows)
system_ap_shift.to_csv(OUTPUT_ROOT / "system_ap_shift_corrected.csv", index=False)


In [ ]:

# ============================================================
# 13 – Kaskade: B0 blockiert, Stage 2 priorisiert Review
# ============================================================

def scores_for_system(model_name, seed):
    dat = np.load(system_score_path(model_name, seed))
    return dat["cal"], dat["iid"], dat["holdout"]

cascade_rankers = [
    "B0_SELF",
    "T0_E2E",
    "DAPT_E2E",
    "BASE_EMB_MLP",
    "DAPT_EMB_MLP",
    "CONTRASTIVE_EMB_MLP",
]

cascade_rows = []

for seed in SEEDS:
    ycal = calibration_df["label"].to_numpy(dtype=int)

    b0_cal, b0_iid, b0_hold = scores_for_system("B0_STRUCT_XGB", seed)
    b0_ops = calibration_operating_points(ycal, b0_cal)

    ranker_cache = {}
    ranker_ops = {}
    for ranker in cascade_rankers:
        source_model = "B0_STRUCT_XGB" if ranker == "B0_SELF" else ranker
        cal_s, iid_s, hold_s = scores_for_system(source_model, seed)
        ranker_cache[ranker] = (iid_s, hold_s)
        ranker_ops[ranker] = calibration_operating_points(ycal, cal_s)

    for target in TARGET_FPRS:
        b0_op = b0_ops[target]

        for scenario in SCENARIOS:
            y, b0_score = scenario_score_view(b0_iid, b0_hold, scenario)
            b0_pred = b0_score >= b0_op["threshold"]
            stage1_negative = ~b0_pred

            tn, fp, fn, tp = confusion_matrix(y, b0_pred.astype(int), labels=[0, 1]).ravel()
            stage1_empirical_fpr = fp / max(fp + tn, 1)
            stage1_recall = tp / max(tp + fn, 1)
            base_fn_mask = (y == 1) & stage1_negative
            n_base_fn = int(base_fn_mask.sum())
            neg_idx = np.flatnonzero(stage1_negative)

            for review_frac in REVIEW_FRACTIONS:
                n_review = max(1, int(math.ceil(review_frac * len(neg_idx))))

                for ranker in cascade_rankers:
                    riid, rhold = ranker_cache[ranker]
                    _, rank_score = scenario_score_view(riid, rhold, scenario)

                    ordered = neg_idx[np.argsort(rank_score[neg_idx])[::-1]]
                    review_idx = ordered[:n_review]

                    phish_review = int((y[review_idx] == 1).sum())
                    benign_review = int((y[review_idx] == 0).sum())
                    rescued = int(base_fn_mask[review_idx].sum())

                    # Eigener FPR-Arbeitspunkt des Rankers wird zusätzlich ausgewiesen.
                    ranker_op = ranker_ops[ranker][target]
                    ranker_pred = rank_score >= ranker_op["threshold"]
                    r_tn, r_fp, r_fn, r_tp = confusion_matrix(
                        y, ranker_pred.astype(int), labels=[0, 1]
                    ).ravel()
                    ranker_empirical_fpr = r_fp / max(r_fp + r_tn, 1)

                    review_assisted_recall_upper = (
                        (tp + rescued) / max(tp + fn, 1)
                    )

                    cascade_rows.append({
                        "seed": seed,
                        "scenario": scenario,
                        "target_fpr": target,

                        "stage1_model": "B0_STRUCT_XGB",
                        "stage1_threshold": b0_op["threshold"],
                        "stage1_calibration_empirical_fpr": b0_op["calibration_empirical_fpr"],
                        "stage1_empirical_fpr": stage1_empirical_fpr,
                        "stage1_fp": int(fp),
                        "stage1_tn": int(tn),
                        "stage1_tp": int(tp),
                        "stage1_fn": int(fn),
                        "stage1_recall": stage1_recall,

                        "ranker": ranker,
                        "ranker_own_threshold": ranker_op["threshold"],
                        "ranker_calibration_empirical_fpr": ranker_op["calibration_empirical_fpr"],
                        "ranker_empirical_fpr_at_own_threshold": ranker_empirical_fpr,

                        "review_fraction_of_stage1_negatives": review_frac,
                        "stage1_negative_n": int(len(neg_idx)),
                        "review_n": int(n_review),
                        "review_fraction_of_all_cases": float(n_review / len(y)),
                        "review_phishing_n": phish_review,
                        "review_benign_n": benign_review,
                        "review_precision": float(phish_review / max(n_review, 1)),
                        "rescued_stage1_fn": rescued,
                        "capture_rate_of_stage1_fn": float(rescued / max(n_base_fn, 1)),
                        "review_assisted_recall_upper_bound": float(review_assisted_recall_upper),
                        "potential_recall_gain_pp": float(
                            100 * (review_assisted_recall_upper - stage1_recall)
                        ),
                    })

cascade = pd.DataFrame(cascade_rows)

# Lift gegenüber B0-Self bei exakt identischer Queuegröße.
self_ref = (
    cascade[cascade["ranker"] == "B0_SELF"][
        ["seed", "scenario", "target_fpr", "review_fraction_of_stage1_negatives",
         "rescued_stage1_fn", "review_precision",
         "review_assisted_recall_upper_bound"]
    ]
    .rename(columns={
        "rescued_stage1_fn": "self_rescued_stage1_fn",
        "review_precision": "self_review_precision",
        "review_assisted_recall_upper_bound": "self_review_assisted_recall_upper_bound",
    })
)

cascade = cascade.merge(
    self_ref,
    on=["seed", "scenario", "target_fpr", "review_fraction_of_stage1_negatives"],
    how="left",
)
cascade["rescued_fn_lift_vs_self"] = (
    cascade["rescued_stage1_fn"] - cascade["self_rescued_stage1_fn"]
)
cascade["review_precision_lift_vs_self"] = (
    cascade["review_precision"] - cascade["self_review_precision"]
)
cascade["recall_upper_bound_lift_vs_self_pp"] = 100 * (
    cascade["review_assisted_recall_upper_bound"]
    - cascade["self_review_assisted_recall_upper_bound"]
)

cascade.to_csv(OUTPUT_ROOT / "cascade_results.csv", index=False)

cascade_summary = (
    cascade
    .groupby(
        ["scenario", "target_fpr", "ranker", "review_fraction_of_stage1_negatives"],
        as_index=False
    )
    .agg(
        stage1_empirical_fpr_mean=("stage1_empirical_fpr", "mean"),
        ranker_empirical_fpr_mean=("ranker_empirical_fpr_at_own_threshold", "mean"),
        stage1_recall_mean=("stage1_recall", "mean"),
        review_precision_mean=("review_precision", "mean"),
        rescued_stage1_fn_mean=("rescued_stage1_fn", "mean"),
        capture_rate_of_stage1_fn_mean=("capture_rate_of_stage1_fn", "mean"),
        review_assisted_recall_upper_bound_mean=("review_assisted_recall_upper_bound", "mean"),
        potential_recall_gain_pp_mean=("potential_recall_gain_pp", "mean"),
        rescued_fn_lift_vs_self_mean=("rescued_fn_lift_vs_self", "mean"),
        recall_upper_bound_lift_vs_self_pp_mean=("recall_upper_bound_lift_vs_self_pp", "mean"),
    )
)
cascade_summary.to_csv(OUTPUT_ROOT / "cascade_summary.csv", index=False)
display(cascade_summary.head(40))


In [ ]:

# ============================================================
# 14 – Fehlerkomplementarität: B0 gegen alle relevanten Modelle
# ============================================================

PAIR_MODELS = [
    "T0_E2E",
    "DAPT_E2E",
    "BASE_EMB_MLP",
    "DAPT_EMB_MLP",
    "CONTRASTIVE_EMB_MLP",
]

error_rows = []

for seed in SEEDS:
    ycal = calibration_df["label"].to_numpy(dtype=int)

    b0_cal, b0_iid, b0_hold = scores_for_system("B0_STRUCT_XGB", seed)
    b0_ops = calibration_operating_points(ycal, b0_cal)

    for candidate in PAIR_MODELS:
        c_cal, c_iid, c_hold = scores_for_system(candidate, seed)
        c_ops = calibration_operating_points(ycal, c_cal)

        for target in TARGET_FPRS:
            b0_op = b0_ops[target]
            c_op = c_ops[target]

            for scenario in SCENARIOS:
                y, b0_score = scenario_score_view(b0_iid, b0_hold, scenario)
                _, c_score = scenario_score_view(c_iid, c_hold, scenario)

                bp = b0_score >= b0_op["threshold"]
                cp = c_score >= c_op["threshold"]

                b_fn = (y == 1) & (~bp)
                c_fn = (y == 1) & (~cp)
                b_fp = (y == 0) & bp
                c_fp = (y == 0) & cp

                b_tn = (y == 0) & (~bp)
                c_tn = (y == 0) & (~cp)

                common_fn = int((b_fn & c_fn).sum())
                rescued_b0_fn = int((b_fn & (~c_fn)).sum())
                regressed_b0_tp = int(((y == 1) & bp & c_fn).sum())

                common_fp = int((b_fp & c_fp).sum())
                avoided_b0_fp = int((b_fp & (~c_fp)).sum())
                new_fp_vs_b0 = int((b_tn & c_fp).sum())

                b0_fpr = float(b_fp.sum() / max((y == 0).sum(), 1))
                cand_fpr = float(c_fp.sum() / max((y == 0).sum(), 1))

                fn_union = int((b_fn | c_fn).sum())
                fp_union = int((b_fp | c_fp).sum())

                error_rows.append({
                    "seed": seed,
                    "scenario": scenario,
                    "target_fpr": target,
                    "base_model": "B0_STRUCT_XGB",
                    "candidate_model": candidate,

                    "base_calibration_empirical_fpr": b0_op["calibration_empirical_fpr"],
                    "candidate_calibration_empirical_fpr": c_op["calibration_empirical_fpr"],
                    "base_empirical_fpr": b0_fpr,
                    "candidate_empirical_fpr": cand_fpr,
                    "candidate_minus_base_fpr": cand_fpr - b0_fpr,

                    "base_fn": int(b_fn.sum()),
                    "candidate_fn": int(c_fn.sum()),
                    "common_fn": common_fn,
                    "rescued_b0_fn": rescued_b0_fn,
                    "regressed_b0_tp": regressed_b0_tp,
                    "fn_jaccard": float(common_fn / max(fn_union, 1)),
                    "fn_rescue_rate": float(rescued_b0_fn / max(b_fn.sum(), 1)),

                    "base_fp": int(b_fp.sum()),
                    "candidate_fp": int(c_fp.sum()),
                    "common_fp": common_fp,
                    "avoided_b0_fp": avoided_b0_fp,
                    "new_fp_vs_b0": new_fp_vs_b0,
                    "fp_jaccard": float(common_fp / max(fp_union, 1)),
                })

error_df = pd.DataFrame(error_rows)
error_df.to_csv(OUTPUT_ROOT / "error_complementarity.csv", index=False)

error_summary = (
    error_df
    .groupby(["scenario", "target_fpr", "candidate_model"], as_index=False)
    .agg(
        base_empirical_fpr_mean=("base_empirical_fpr", "mean"),
        candidate_empirical_fpr_mean=("candidate_empirical_fpr", "mean"),
        candidate_minus_base_fpr_mean=("candidate_minus_base_fpr", "mean"),
        base_fn_mean=("base_fn", "mean"),
        candidate_fn_mean=("candidate_fn", "mean"),
        rescued_b0_fn_mean=("rescued_b0_fn", "mean"),
        regressed_b0_tp_mean=("regressed_b0_tp", "mean"),
        fn_rescue_rate_mean=("fn_rescue_rate", "mean"),
        fn_jaccard_mean=("fn_jaccard", "mean"),
        base_fp_mean=("base_fp", "mean"),
        candidate_fp_mean=("candidate_fp", "mean"),
        new_fp_vs_b0_mean=("new_fp_vs_b0", "mean"),
        avoided_b0_fp_mean=("avoided_b0_fp", "mean"),
        fp_jaccard_mean=("fp_jaccard", "mean"),
    )
)
error_summary.to_csv(OUTPUT_ROOT / "error_complementarity_summary.csv", index=False)
display(error_summary.head(40))


In [ ]:

# ============================================================
# 15 – Fehlerprofile und qualitative Beispiele (Seed 42)
# ============================================================

def enrich_frame(frame):
    f = frame.reset_index(drop=True).copy()
    f["text_chars"] = f["text"].fillna("").astype(str).str.len()
    f["url_chars"] = (
        f["url"].fillna("").astype(str).str.len()
        if "url" in f.columns else np.nan
    )
    return f

feature_rows = []
sample_rows = []
seed = 42
ycal = calibration_df["label"].to_numpy(dtype=int)

b0_cal, b0_iid, b0_hold = scores_for_system("B0_STRUCT_XGB", seed)
b0_ops = calibration_operating_points(ycal, b0_cal)

for candidate in ["T0_E2E", "DAPT_E2E", "DAPT_EMB_MLP", "CONTRASTIVE_EMB_MLP"]:
    c_cal, c_iid, c_hold = scores_for_system(candidate, seed)
    c_ops = calibration_operating_points(ycal, c_cal)

    for target in TARGET_FPRS:
        for scenario in SCENARIOS:
            source, idx = SCENARIOS[scenario]
            frame = iid_df.iloc[idx] if source == "iid" else holdout_df.iloc[idx]
            frame = enrich_frame(frame)

            y, b0_score = scenario_score_view(b0_iid, b0_hold, scenario)
            _, c_score = scenario_score_view(c_iid, c_hold, scenario)

            bp = b0_score >= b0_ops[target]["threshold"]
            cp = c_score >= c_ops[target]["threshold"]

            base_fn = (y == 1) & (~bp)
            rescued = base_fn & cp
            common_fn = base_fn & (~cp)

            for group_name, mask in [
                ("B0_FN_RESCUED_BY_CANDIDATE", rescued),
                ("COMMON_FN", common_fn),
            ]:
                pos = np.flatnonzero(mask)
                if len(pos):
                    for feature in ["text_chars", "url_chars"]:
                        vals = pd.to_numeric(frame.iloc[pos][feature], errors="coerce")
                        if vals.notna().any():
                            feature_rows.append({
                                "seed": seed,
                                "candidate_model": candidate,
                                "scenario": scenario,
                                "target_fpr": target,
                                "group": group_name,
                                "feature": feature,
                                "n": int(vals.notna().sum()),
                                "mean": float(vals.mean()),
                                "median": float(vals.median()),
                                "q25": float(vals.quantile(0.25)),
                                "q75": float(vals.quantile(0.75)),
                            })

            # Je Kombination maximal fünf repräsentative gerettete B0-FN.
            rescued_idx = np.flatnonzero(rescued)
            if len(rescued_idx):
                # Kandidatenscore absteigend: besonders klare Rescue-Fälle.
                rescued_idx = rescued_idx[np.argsort(c_score[rescued_idx])[::-1]][:5]
                for i in rescued_idx:
                    r = frame.iloc[int(i)]
                    sample_rows.append({
                        "seed": seed,
                        "candidate_model": candidate,
                        "scenario": scenario,
                        "target_fpr": target,
                        "base_empirical_fpr": float(
                            ((y == 0) & bp).sum() / max((y == 0).sum(), 1)
                        ),
                        "candidate_empirical_fpr": float(
                            ((y == 0) & cp).sum() / max((y == 0).sum(), 1)
                        ),
                        "b0_score": float(b0_score[i]),
                        "candidate_score": float(c_score[i]),
                        "sha256": str(r.get("sha256", "")),
                        "domain": str(r.get("domain", "")),
                        "url": str(r.get("url", ""))[:500],
                        "template_hash": str(r.get("template_hash", "")),
                        "near_duplicate_to_development": r.get(
                            "near_duplicate_to_development", np.nan
                        ),
                        "min_simhash_distance_to_development": r.get(
                            "min_simhash_distance_to_development", np.nan
                        ),
                        "text_preview": str(r.get("text", ""))[:400].replace("\n", " "),
                    })

pd.DataFrame(feature_rows).to_csv(
    OUTPUT_ROOT / "error_feature_profiles_seed42.csv", index=False
)
pd.DataFrame(sample_rows).to_csv(
    OUTPUT_ROOT / "rescued_b0_false_negatives_seed42.csv", index=False
)

print({
    "feature_profile_rows": len(feature_rows),
    "qualitative_rescue_rows": len(sample_rows),
})


In [ ]:

# ============================================================
# 16 – Kernübersichten für die spätere Bachelorarbeit
# ============================================================

# A) FINAL FREEZE: 25 % Labels, primärer 0,5%-FPR-Punkt
freeze_25_primary = freeze_summary[
    (freeze_summary["label_budget"] == 0.25)
    & (freeze_summary["target_fpr"] == PRIMARY_TARGET_FPR)
].copy()
freeze_25_primary.to_csv(
    OUTPUT_ROOT / "TABLE_freeze_25pct_primary_fpr.csv", index=False
)

# B) Systemvergleich: alle drei FPR-Punkte, Stressszenarien
system_stress_table = system_summary[
    system_summary["scenario"].isin([
        "TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
    ])
].copy()
system_stress_table.to_csv(
    OUTPUT_ROOT / "TABLE_system_stress_all_fpr.csv", index=False
)

# C) Kaskade: Stressszenarien, alle drei FPR-Punkte
cascade_stress = cascade_summary[
    cascade_summary["scenario"].isin([
        "TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
    ])
].copy()
cascade_stress.to_csv(
    OUTPUT_ROOT / "TABLE_cascade_stress_all_fpr.csv", index=False
)

# D) Fehleranalyse: Stressszenarien
error_stress = error_summary[
    error_summary["scenario"].isin([
        "TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
    ])
].copy()
error_stress.to_csv(
    OUTPUT_ROOT / "TABLE_error_complementarity_stress.csv", index=False
)

print("Bachelor-Tabellen exportiert.")


In [ ]:

# ============================================================
# 17 – Completion Audit und ZIP
# ============================================================

# Erwartete korrigierte Freeze-Zeilen:
# 3 Reps × 3 Classifier × 5 Seeds × 3 Budgets × 6 Szenarien × 3 FPR
expected_freeze_rows = (
    len(REPRESENTATIONS) * len(CLASSIFIERS) * len(SEEDS)
    * len(LABEL_BUDGETS) * len(SCENARIOS) * len(TARGET_FPRS)
)
if len(freeze_results) != expected_freeze_rows:
    raise RuntimeError(
        f"freeze_corrected_results unvollständig: {len(freeze_results)} / "
        f"{expected_freeze_rows}"
    )

expected_system_rows = (
    len(SYSTEM_MODELS) * len(SEEDS) * len(SCENARIOS) * len(TARGET_FPRS)
)
if len(system_results) != expected_system_rows:
    raise RuntimeError(
        f"system_results unvollständig: {len(system_results)} / {expected_system_rows}"
    )

config = {
    "status": "COMPLETE",
    "purpose": "FINAL FREEZE extension: FPR + AP correction + 25pct systems/cascade/error analysis",
    "seeds": SEEDS,
    "label_budgets": LABEL_BUDGETS,
    "system_label_budget": SYSTEM_LABEL_BUDGET,
    "target_fprs": TARGET_FPRS,
    "primary_target_fpr": PRIMARY_TARGET_FPR,
    "scenarios": list(SCENARIOS.keys()),
    "ap_primary_reference": "IID_BALANCED (50/50 prevalence matched)",
    "ap_equal_n_sensitivity_n": int(2 * n_each_iid),
    "system_models": SYSTEM_MODELS,
    "e2e": {
        "epochs": E2E_EPOCHS,
        "lr": E2E_LR,
        "weight_decay": E2E_WEIGHT_DECAY,
        "warmup_ratio": E2E_WARMUP_RATIO,
        "max_length": MAX_LENGTH,
        "dapt_pretraining_repeated": False,
    },
    "holdout_used_for_training": False,
    "holdout_used_for_threshold_selection": False,
}

(OUTPUT_ROOT / "HYBRID_FREEZE_EXTENSION_COMPLETE.json").write_text(
    json.dumps(config, indent=2), encoding="utf-8"
)

archive = shutil.make_archive(
    "/kaggle/working/phreshphish_hybrid_freeze_extension",
    "zip",
    root_dir=OUTPUT_ROOT,
)

print(json.dumps(config, indent=2))
print({"zip": archive})



## Zentrale Ergebnisdateien

### AP-/FPR-Korrektur des FINAL FREEZE
- `testset_prevalence_audit.csv`
- `freeze_corrected_results.csv`
- `freeze_corrected_summary.csv`
- `ap_shift_corrected_prevalence_matched.csv`
- `ap_equal_n_sensitivity_results.csv`
- `ap_shift_equal_n_sensitivity.csv`

### Systemvergleich bei 25 % Labels
- `system_25pct_results.csv`
- `system_25pct_summary.csv`
- `system_ap_shift_corrected.csv`
- `b0_train_only_tuning.csv`

### Kaskade
- `cascade_results.csv`
- `cascade_summary.csv`

### Fehleranalyse
- `error_complementarity.csv`
- `error_complementarity_summary.csv`
- `error_feature_profiles_seed42.csv`
- `rescued_b0_false_negatives_seed42.csv`

### Direkt für Kapitel 5
- `TABLE_freeze_25pct_primary_fpr.csv`
- `TABLE_system_stress_all_fpr.csv`
- `TABLE_cascade_stress_all_fpr.csv`
- `TABLE_error_complementarity_stress.csv`

### Abschluss
- `HYBRID_FREEZE_EXTENSION_COMPLETE.json`
- `/kaggle/working/phreshphish_hybrid_freeze_extension.zip`

**Interpretationsregel:**  
`IID_ORIGINAL` bleibt für operative Kennzahlen erhalten. Für Aussagen wie
„AP verschlechtert sich unter OOD um …“ wird nur `IID_BALANCED` als Referenz
verwendet. Die Equal-N-Auswertung ist eine zusätzliche Sensitivitätsprüfung.
